In [20]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [21]:
load_dotenv(override=True)


True

In [13]:
# Let's just check emails are working for you

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("i77tag+sendgrid@gmail.com")
    to_email = To("i77tag+sendgrid@gmail.com")
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

202


### Agent Workflow

In [14]:
instruction1 = "You are a sales agent working for ComplAI, \
    a company that provides a SaaS tool for ensuring SOC2 compliance and preparign for audits, powerd by AI, \
    You write Professional, serious cold emails."

instruction2 = "You are a humorous, engaging sales agent working for ComplAI, \
    a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
    You write witty, engaging cold emails that are likely to get a response."

instruction3 = "You are a busy sales agent working for ComplAI, \
    Company that provides SaaS tools for ensuring SOC2 compliance and preparing for audits, powered by AI. \
    You write concise, to the point emails."

In [15]:
sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instruction1,
    model="gpt-4o-mini"
)
sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instruction2,
    model="gpt-4o-mini"
)
sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instruction3,
    model="gpt-4o-mini"
)

In [16]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Enhance Your SOC 2 Compliance with AI-Powered Solutions

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m reaching out from ComplAI, where we specialize in helping organizations like yours streamline their SOC 2 compliance processes.

As you know, maintaining compliance is not just a regulatory requirement but a vital component of building trust with your clients. Our AI-driven platform simplifies the compliance journey, automating documentation, risk assessments, and continuous monitoring. This not only saves time but also reduces the risk of oversight during audits.

Here’s how our solution can benefit your organization:

1. **Automated Compliance Tracking**: Keep up-to-date with the latest requirements without manual hassle.
2. **Risk Assessment Tools**: Identify vulnerabilities before they become issues.
3. **Audit Preparation**: Simplified documentation management to streamline your audit process.

I would love the opportunity to

In [22]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: Streamline Your SOC 2 Compliance Efforts with ComplAI

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m reaching out to introduce you to ComplAI, a cutting-edge SaaS solution designed to simplify your SOC 2 compliance and audit preparation processes.

In today’s regulatory environment, maintaining compliance can be a daunting task. ComplAI leverages advanced AI technology to provide real-time insights and streamlined workflows, making it easier to meet compliance requirements and prepare for audits efficiently. 

Here are a few ways ComplAI can benefit your organization:

- **Automated Risk Assessments:** Identify and address potential compliance gaps before they become issues.
- **Centralized Documentation:** Store and manage all compliance-related documentation in one secure location.
- **Real-Time Monitoring:** Continuously track your compliance status and receive alerts on any changes.

Many of our clients have seen a significan

In [ ]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the give option. \
        Imagine you are a customer and pick the one you are most likely to respond to. \
        Do not give an explanation; reply with the selected email only."
)

In [27]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

    outputs = [result.final_output for result in results]

    emails = "Cold sales emails: \n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email: \n{best.final_output}")

Best sales email: 
Subject: Simplify Your SOC 2 Compliance Journey with ComplAI

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m reaching out from ComplAI, where we specialize in streamlining SOC 2 compliance through our advanced SaaS platform.

Navigating the complexities of SOC 2 requirements can be daunting, especially with the increasing scrutiny on data security and privacy. Our AI-powered solution not only automates the compliance process but also prepares you for audits effectively, saving your team both time and resources.

Many of our clients have seen a significant reduction in the time spent on compliance tasks while enhancing their data security posture. We would love the opportunity to discuss how ComplAI can specifically address the challenges your organization may face.

Would you be open to a brief call next week? I can provide a tailored demonstration to showcase how our platform can support your compliance goals.

Thank you 

### Part2: use of tools

In [28]:
sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instruction1,
    model="gpt-4o-mini"
)
sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instruction2,
    model="gpt-4o-mini"
)
sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instruction3,
    model="gpt-4o-mini"
)

In [29]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a sales agent working for ComplAI,     a company that provides a SaaS tool for ensuring SOC2 compliance and preparign for audits, powerd by AI,     You write Professional, serious cold emails.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [30]:
@function_tool
def send_email(body: str):
    """ Send out email with the given  body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("i77tag+sendgrid@gmail.com")
    to_email = To("tismailov@deloitte.com")
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "sales_email", content).get()
    response = sg.client.mail.send.post(reques_body=mail)
    return {"status": "success"}

In [31]:
send_email

FunctionTool(name='send_email', description='Send out email with the given  body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115f0e3e0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [32]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11727c400>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [33]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3]
tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11727e980>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1177bc180>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', description='Write 

### Creat a planning agent

In [35]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""

sales_manager = Agent(name="Sales manger", instructions=instructions, tools=tools, model="gpt-4o-mini")

message = "Send cold sales email addressed to 'Dear SEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)